# Eksplorasi Online Retail II
Jalankan notebook ini dengan Python 3.12 dari folder proyek atau folder notebook. Mulai dari fakta Parquet yang tersedia di paket. Untuk membuatnya kembali dari data Excel, ikuti README.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src/pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.reference import OUT, rupiah
from src.validation import validate

## Lihat struktur fakta penjualan
Amati tipe kolom, nilai kosong, dan contoh baris. Periksa mengapa satu invoice dapat muncul beberapa kali.

In [ ]:
sales = pd.read_parquet(OUT / "fact_sales.parquet")
print("Bentuk tabel:", sales.shape)
display(sales.head())
display(sales.dtypes)
display(sales.isna().sum().sort_values(ascending=False))

## Hitung penjualan dan transaksi per negara
Nominal IDR merupakan konversi analitis. Hitung invoice unik dengan `nunique`.

In [ ]:
wilayah = sales.groupby("CountryLabel").agg(
    penjualan=("RevenueIDR", "sum"),
    transaksi=("Invoice", "nunique"),
).sort_values("penjualan", ascending=False).head(10)
display(wilayah.assign(penjualan=wilayah.penjualan.map(rupiah)))

## Rekonsiliasi hasil
Keluaran berikut membandingkan KPI pandas dari pipeline dengan hasil kueri yang dihitung ulang di DuckDB.

In [ ]:
import json
quality = json.loads((OUT / "data_quality_report.json").read_text(encoding="utf-8"))
display(quality["eksklusi_bertahap"])
display(validate(save=False)["pemeriksaan_kpi"])